# Lecture 10 · Multi-agent systems: when a team of agents pays (Fable edition)

Companion notebook, generated from the code on the lecture pages (https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/). Run the cells in order; each one corresponds to a numbered step on the pages. Steps marked 💳 need `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `LLM_MODEL`) in the environment and are skipped otherwise.

## Step 1 — The assistant as one agent, and what its tools cost

[1. Decide whether one agent is enough](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/01-decide-whether-one-agent-is-enough/)

In [ ]:
# The assistant as one agent: its data, its seven tools, and what the
# tool catalogue costs in every request.
import itertools, json, os, random, re, sys, time, pathlib, urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import Literal
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

SITE = "https://aiml-study-guides.onrender.com"
def fetch(name, url):
    path = pathlib.Path(name)
    if not path.exists():
        urllib.request.urlretrieve(SITE + url, path)
    return [json.loads(line) for line in path.open(encoding="utf-8")]

# Frozen course assets: Lecture 8's questions (same split), the notes,
# and this lecture's labelled requests.
rows = fetch("l8-questions.jsonl",
             "/datasets/generative-agentic-ai/l8-questions.jsonl")
notes = [n for n in fetch(
    "lectures-01-06.en.jsonl",
    "/corpus/generative-agentic-ai/lectures-01-06.en.jsonl") if n["heading"]]
requests = fetch("l10-requests.jsonl",
                 "/datasets/generative-agentic-ai/l10-requests.jsonl")
for i, r in enumerate(rows):
    r["family"] = r["section"] if r["lecture"] else f"none-{i}"
random.seed(0)
families = sorted({r["family"] for r in rows}); random.shuffle(families)
fifth = len(families) // 5
test_f, val_f = set(families[:fifth]), set(families[fifth:2 * fifth])
val = [r for r in rows if r["family"] in val_f]       # 145 questions

# Lecture 7's retriever, as Lecture 9 used it: TF-IDF over the notes.
vec = TfidfVectorizer(stop_words="english").fit([n["text"] for n in notes])
doc_m = vec.transform([n["text"] for n in notes])
def retrieve(question, k=3):
    scores = (doc_m @ vec.transform([question]).T).toarray().ravel()
    return [notes[i] for i in scores.argsort()[::-1][:k]]

NUMBERED = re.compile(r"\d+\.\d+ ")      # "2.1 Zero-shot, few-shot…"

# What every tool may read: the student, their list, their outbox, today.
TODAY = date(2026, 11, 5)                     # fixed, so runs repeat
FIRST_TASKS = [("t1", "finish the Lecture 6 homework"),
               ("t2", "revise attention"), ("t3", "buy bread"),
               ("t4", "read the DPO paper")]
@dataclass
class Student:
    student_id: str = "student-42"
    today: date = TODAY
    tasks: dict = field(default_factory=lambda: dict(FIRST_TASKS))
    outbox: list = field(default_factory=list)
    new_ids: object = field(default_factory=lambda: itertools.count(
        len(FIRST_TASKS) + 1))       # t5, t6, …: an id is never reused

def route_offline(question):     # offline stand-in: the top note's lecture
    return retrieve(question, 1)[0]["lecture"]
ROUTE = route_offline            # the hosted router replaces it below

# The assistant's seven tools: Lectures 6 and 9, now with ToolRuntime.
@tool
def search_notes(query: str) -> list[dict]:
    """Search the course notes (Lectures 1-6) for a topic.

    Use it when the student asks what the course says about something.
    Returns three sections, each with its lecture, numbered heading and
    first words. Cite them as (Lecture N, M.K).
    """
    found = [n for n in retrieve(query, 10) if NUMBERED.match(n["heading"])]
    return [{"lecture": n["lecture"], "heading": n["heading"],
             "text": " ".join(n["text"].split()[:60])} for n in found[:3]]

@tool
def route_question(question: str) -> int:
    """Name the lecture (1-6) that answers a student's question, or 0.

    Use it when the student asks which lecture to read, not for the
    content itself.
    """
    return ROUTE(question)

@tool
def days_until(deadline: date, runtime: ToolRuntime[Student]) -> int:
    """Count whole days from today to a deadline (YYYY-MM-DD)."""
    return (deadline - runtime.context.today).days

@tool
def add_task(title: str, runtime: ToolRuntime[Student]) -> str:
    """Add one task to the student's to-do list. Returns its id."""
    task_id = f"t{next(runtime.context.new_ids)}"
    runtime.context.tasks[task_id] = title
    return f"Added «{title}» (id {task_id})."

@tool
def list_tasks(runtime: ToolRuntime[Student]) -> str:
    """List the student's open tasks with their ids."""
    tasks = runtime.context.tasks
    return "\n".join(f"{k}: {v}" for k, v in tasks.items()) or "No tasks."

@tool
def delete_task(task_id: str, runtime: ToolRuntime[Student]) -> str:
    """Delete one task by its id. Look the id up with list_tasks first."""
    title = runtime.context.tasks.pop(task_id, None)
    if title is None:
        return f"No task with id {task_id}. Call list_tasks for the ids."
    return f"Deleted «{title}»."

@tool
def send_reminder(text: str, when: str,
                  runtime: ToolRuntime[Student]) -> str:
    """Email the student a reminder at a time (YYYY-MM-DD HH:MM)."""
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}", when):
        return "Error: give the time as YYYY-MM-DD HH:MM."
    runtime.context.outbox.append({"text": text, "when": when})
    return f"Reminder set for {when}: {text}"

NOTES_TOOLS = [search_notes, route_question]
TODO_TOOLS = [days_until, add_task, list_tasks, delete_task, send_reminder]
ALL_TOOLS = NOTES_TOOLS + TODO_TOOLS

def approx_tokens(obj) -> int:
    """Rough token count: characters / 4 (Step 2 measures real ones)."""
    text = obj if isinstance(obj, str) else json.dumps(obj, default=str)
    return len(text) // 4

def spec(t):
    return convert_to_openai_tool(t)["function"]

print(f"{'tool':<16}{'≈tokens':>8}  parameters the model sees")
for t in ALL_TOOLS:
    print(f"{t.name:<16}{approx_tokens(spec(t)):>8}  "
          f"{list(spec(t)['parameters']['properties'])}")
catalogue = sum(approx_tokens(spec(t)) for t in ALL_TOOLS)
print(f"all seven: ≈{catalogue} tokens, sent with every model call")

# The ScriptedModel of Lecture 6: replays prepared replies, no key.
# A reply is a text, a list of (tool, args), or a function of the
# messages that returns one of those.
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult

CALL_IDS = itertools.count(1)        # tool-call ids, unique in a session

class ScriptedModel(BaseChatModel):
    replies: list
    calls: int = 0
    seen: list = []               # tokens of the messages in each call

    def _generate(self, messages, stop=None, run_manager=None, **kw):
        reply = self.replies[min(self.calls, len(self.replies) - 1)]
        self.calls += 1
        self.seen.append(sum(approx_tokens(str(m.content)) + approx_tokens(
            getattr(m, "tool_calls", None) or "") for m in messages))
        if callable(reply):
            reply = reply(messages)
        if isinstance(reply, str):
            message = AIMessage(reply)
        else:
            message = AIMessage("", tool_calls=[
                {"name": n, "args": a, "id": f"call_{next(CALL_IDS)}"}
                for n, a in reply])
        return ChatResult(generations=[ChatGeneration(message=message)])

    def bind_tools(self, tools, **kwargs):
        return self

    @property
    def _llm_type(self) -> str:
        return "scripted"

def last_tool_text(messages):         # what the last tool returned
    return next(str(m.content) for m in reversed(messages)
                if isinstance(m, ToolMessage))

# The hosted model for the 💳 steps: settings from the environment.
LLM_READY = bool(os.environ.get("LLM_API_KEY"))
MODEL = os.environ.get("LLM_MODEL", "gpt-6-luna")
PRICE_IN, PRICE_OUT = 0.10, 0.50     # $ per 1M tokens, checked 2026-11-01
def chat(**extra):
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model=MODEL, api_key=os.environ["LLM_API_KEY"],
                      base_url=os.environ.get("LLM_BASE_URL"),
                      use_responses_api=True, reasoning={"effort": "none"},
                      max_retries=4, **extra)
hosted = chat() if LLM_READY else None

# Lecture 5's router, unchanged: all section headings, three retrieved
# notes, a strict schema (0.91 on the 145 validation questions).
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
class Route(BaseModel):
    """The lecture of the course notes that answers the question."""
    lecture: Literal[0, 1, 2, 3, 4, 5, 6] = Field(
        description="1–6: the lecture that answers it; 0: no lecture does")
NAMES = {1: "Generative and agentic AI", 2: "Training language models",
         3: "Choosing and configuring models", 4: "Context engineering",
         5: "LangChain", 6: "Agents and tools"}
headings = defaultdict(list)
for n in notes:
    headings[n["lecture"]].append(n["heading"])
HEADINGS = "\n".join(f"{k}: {v}\n  " + "\n  ".join(headings[k])
                     for k, v in NAMES.items())
TASK = ("You route a student's question to the lecture of the course "
        "notes that answers it. Return lecture 0 if no lecture covers "
        "it.\n\nLectures:\n")
def with_notes(question, k=3):
    return "\n\n".join(
        f"[Lecture {n['lecture']}] {n['heading']}\n"
        + " ".join(n["text"].split()[:250]) for n in retrieve(question, k))
route_prompt = ChatPromptTemplate.from_messages([
    ("system", TASK + "{headings}"),
    ("human", "Notes that may help:\n{notes}\n\nQuestion: {question}")])
def make_router(model):
    return (RunnablePassthrough.assign(
                headings=lambda q: HEADINGS,
                notes=lambda q: with_notes(q["question"]))
            | route_prompt | model.with_structured_output(Route)
            | RunnableLambda(lambda r: r.lecture))
if LLM_READY:
    router = make_router(hosted)
    ROUTE = lambda question: router.invoke({"question": question})
print("hosted model:", MODEL if LLM_READY
      else "not configured; the 💳 steps will be skipped")
kinds = Counter(r.get("kind", r["set"]) for r in requests)
print(f"requests: {len(requests)} ({dict(kinds)})")

## Step 2 — Tool selection with 7 and 28 tools, and four fixes 💳 API

[1. Decide whether one agent is enough](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/01-decide-whether-one-agent-is-enough/)

In [ ]:
# 💳 Tool selection as the catalogue grows to 28 tools, and four fixes
# that keep a single agent: descriptions, namespaces, tool search.
import threading
from concurrent.futures import ThreadPoolExecutor
from pydantic import create_model
from langchain_core.tools import StructuredTool
from langchain_core.callbacks import BaseCallbackHandler
from langchain.agents import create_agent
from langchain.agents.middleware import (
    wrap_model_call, LLMToolSelectorMiddleware)

# 21 tools a grown assistant gains from other apps: (name, namespaced
# name, parameters, short description, clear description). A clear one
# says what the tool does, when to use it, and which neighbour to use
# instead; {names} in it follow the configuration's naming.
S = str
EXTRA = [
 ("search_my_notes", "mynotes_search", {"query": S}, "Search notes.",
  "Search the student's own personal notes. Not for the course "
  "material: use {search_notes}."),
 ("search_forum", "forum_search", {"query": S}, "Search the forum.",
  "Search posts by other students on the course forum. Not for the "
  "course notes: use {search_notes}."),
 ("search_web", "web_search", {"query": S}, "Search the web.",
  "Search the public web. Use only when the course cannot help."),
 ("get_syllabus", "course_syllabus", {}, "Get the syllabus.",
  "List all lectures with their titles. To find the lecture for one "
  "question, use {route_question}."),
 ("find_lecture_video", "course_find_video", {"lecture": int},
  "Find a lecture.", "Return the link to a lecture's video recording, "
  "by lecture number. Not for which lecture covers a topic: use "
  "{route_question}."),
 ("suggest_next_lecture", "course_suggest_next", {},
  "Suggest a lecture.", "Suggest the lecture to study next from the "
  "student's progress. Not for routing a question: use "
  "{route_question}."),
 ("get_deadline", "course_get_deadline", {"assignment": S},
  "Get a deadline.", "Look up the official due date of a course "
  "assignment by name. To count days to a known date use "
  "{days_until}."),
 ("current_time", "time_now", {}, "Current time.",
  "Return the time of day. Today's date is already in the system "
  "prompt."),
 ("count_study_days", "time_count_study_days", {"until": date},
  "Count days.", "Count study days, Monday to Friday, from today to a "
  "date. For calendar days use {days_until}."),
 ("add_note", "mynotes_add", {"text": S}, "Add a note.",
  "Save a personal note for the student. For things to do use "
  "{add_task}."),
 ("create_flashcard", "flashcards_create", {"front": S, "back": S},
  "Create a card.", "Create a revision flashcard: a question and its "
  "answer."),
 ("add_calendar_event", "calendar_add_event", {"title": S, "start": S},
  "Add an event.", "Add an event with a start time to the student's "
  "calendar. For a task without a time use {add_task}."),
 ("list_notes", "mynotes_list", {}, "List notes.",
  "List the student's personal notes. For tasks use {list_tasks}."),
 ("list_flashcards", "flashcards_list", {}, "List cards.",
  "List the student's revision flashcards."),
 ("list_calendar_events", "calendar_list_events", {}, "List events.",
  "List upcoming calendar events. For to-do tasks use {list_tasks}."),
 ("delete_note", "mynotes_delete", {"note_id": S}, "Delete a note.",
  "Delete one personal note by id. For tasks use {delete_task}."),
 ("delete_calendar_event", "calendar_delete_event", {"event_id": S},
  "Delete an event.", "Delete one calendar event by id. For tasks use "
  "{delete_task}."),
 ("archive_task", "todo_archive_task", {"task_id": S}, "Archive a task.",
  "Mark a task done and keep it in the history. To remove a task "
  "completely use {delete_task}."),
 ("send_email", "email_send", {"to": S, "subject": S, "body": S},
  "Send an email.", "Email another person in the student's name. For a "
  "reminder to the student use {send_reminder}."),
 ("post_to_forum", "forum_post", {"title": S, "body": S},
  "Post to the forum.", "Publish a post on the course forum in the "
  "student's name."),
 ("schedule_push_notification", "push_schedule", {"text": S, "when": S},
  "Schedule a notification.", "Send a push notification to the "
  "student's phone at a time. For an email reminder use "
  "{send_reminder}."),
]
CORE = {  # the seven tools: namespaced name and clear description
 "search_notes": ("course_search_notes", "Search the course notes "
  "(Lectures 1-6) and return three sections. Use when the student asks "
  "what the course says. Not for the student's own notes "
  "({search_my_notes}) or for which lecture to read "
  "({route_question})."),
 "route_question": ("course_route_question", "Name the lecture (1-6) "
  "that answers a question, or 0. Use when the student asks which "
  "lecture to read. For the content use {search_notes}."),
 "days_until": ("time_days_until", "Count calendar days from today to "
  "a date (YYYY-MM-DD). Not for study days ({count_study_days}) or an "
  "assignment's due date ({get_deadline})."),
 "add_task": ("todo_add_task", "Add one task to the to-do list; returns "
  "its id. For things to do without a time. Not for notes "
  "({add_note}) or timed events ({add_calendar_event})."),
 "list_tasks": ("todo_list_tasks", "List the open to-do tasks with "
  "their ids. Not for notes or calendar events."),
 "delete_task": ("todo_delete_task", "Delete one to-do task completely, "
  "by id. To mark it done and keep it use {archive_task}."),
 "send_reminder": ("todo_send_reminder", "Email the student a reminder "
  "at a time (YYYY-MM-DD HH:MM). Not for phone notifications "
  "({schedule_push_notification}) or emails to others "
  "({send_email})."),
}

def stub(name, description, schema):
    return StructuredTool.from_function(
        func=lambda **kw: "not run in this step", name=name,
        description=description, args_schema=schema)

def catalogue(n_tools, clear=False, spaced=False):
    """The tools of one configuration, as stubs (never run here)."""
    alias = {t.name: CORE[t.name][0] if spaced else t.name
             for t in ALL_TOOLS}
    alias |= {e[0]: e[1] if spaced else e[0] for e in EXTRA}
    out = [stub(alias[t.name],
                CORE[t.name][1].format(**alias) if clear else t.description,
                t.tool_call_schema) for t in ALL_TOOLS]
    for name, _, params, short, long in EXTRA[:n_tools - len(out)]:
        schema = create_model(name, **{k: (v, ...) for k, v in
                                       params.items()})
        out.append(stub(alias[name], long.format(**alias) if clear
                        else short, schema))
    return out, {v: k for k, v in alias.items()}  # back to plain names

# Search by words: TF-IDF with a crude stemmer, so "tasks" finds "task".
def stems(text):
    return [w[:-1] if w.endswith("s") and len(w) > 3 else w
            for w in re.findall(r"[a-z]+", text.lower())]
def word_search(docs, query, k):
    v = TfidfVectorizer(analyzer=stems).fit(docs + [query])
    s = (v.transform(docs) @ v.transform([query]).T).toarray().ravel()
    return list(s.argsort()[::-1][:k])

def top_tools(tools, query, k=5):     # over the names and descriptions
    docs = [t.name.replace("_", " ") + " " + t.description for t in tools]
    return [tools[i] for i in word_search(docs, query, k)]

@wrap_model_call
def tool_search(request, handler):
    question = str(request.messages[-1].content)
    return handler(request.override(tools=top_tools(request.tools,
                                                    question)))

CONFIGS = {   # name: (tools, clear descriptions, namespaces, middleware)
    "7 tools":            (7, False, False, []),
    "28 tools":           (28, False, False, []),
    "+ clear text":       (28, True, False, []),
    "+ namespaces":       (28, True, True, []),
    "+ search by words":  (28, True, True, [tool_search]),
    "+ model picks 5":    (28, True, True, "selector"),
}
tool_reqs = [r for r in requests if r["set"] == "tools"]
print("configuration      tools sent per call   ≈tokens")
for name, (n, clear, spaced, mw) in CONFIGS.items():
    tools, _ = catalogue(n, clear, spaced)
    sent = tools[:5] if mw else tools       # a search sends 5 of the 28
    print(f"{name:<19}{len(sent):>6} of {len(tools):<12}"
          f"{sum(approx_tokens(spec(t)) for t in sent):>8}")

# Offline: how often does search by words keep the right tool in its 5?
tools, back = catalogue(28, True, True)
kept = [back[t.name] for r in tool_reqs
        for t in top_tools(tools, r["request"]) if back[t.name] == r["tool"]]
print(f"search by words keeps the right tool in its 5 for "
      f"{len(kept)}/{len(tool_reqs)} requests")

class Meter(BaseCallbackHandler):
    """Counts the model calls and tokens of everything run under it."""
    def __init__(self):
        self.calls = self.tokens_in = self.tokens_out = 0
        self.lock = threading.Lock()
    def on_chat_model_start(self, serialized, messages, **kw):
        with self.lock:
            self.calls += 1
    def on_llm_end(self, response, **kw):
        for gen in response.generations:
            for g in gen:
                u = getattr(g.message, "usage_metadata", None) or {}
                with self.lock:
                    self.tokens_in += u.get("input_tokens", 0)
                    self.tokens_out += u.get("output_tokens", 0)
    def cost(self):
        return (self.tokens_in * PRICE_IN
                + self.tokens_out * PRICE_OUT) / 1e6

def in_parallel(label, fn, items, workers=8):
    """Run fn over items in threads; print progress at half and end."""
    out, t0, done = [None] * len(items), time.time(), [0]
    def one(i):
        out[i] = fn(items[i])
        with lock:
            done[0] += 1
            if done[0] in (len(items) // 2, len(items)):
                print(f"    {label}: {done[0]}/{len(items)} in "
                      f"{time.time() - t0:.0f} s", flush=True)
    lock = threading.Lock()
    with ThreadPoolExecutor(workers) as pool:
        list(pool.map(one, range(len(items))))
    return out

SYSTEM = (f"You are a study assistant. Today is "
          f"{TODAY:%A, %Y-%m-%d}. Use the tools.")
def first_choice(config, run):
    """The tool names in the model's first reply, for every request."""
    n, clear, spaced, mw = CONFIGS[config]
    tools, back = catalogue(n, clear, spaced)
    if mw == "selector":
        mw = [LLMToolSelectorMiddleware(max_tools=5)]
    agent = create_agent(hosted, tools, system_prompt=SYSTEM, middleware=mw)
    def one(r):
        meter = Meter()
        try:
            state = agent.invoke(
                {"messages": [HumanMessage(r["request"])]},
                {"callbacks": [meter]}, interrupt_before=["tools"])
            calls = state["messages"][-1].tool_calls
            names = [back.get(c["name"], c["name"]) for c in calls]
        except Exception as e:                 # a failed call is a miss
            names = [f"error: {type(e).__name__}"]
        return names, meter
    return in_parallel(f"{config}, run {run}", one, tool_reqs)

def right(r, names):
    """The first reply calls the labelled tool, or an allowed first step
    (list_tasks before a deletion, as delete_task's description asks)."""
    return bool({r["tool"], *r["also"]} & set(names))

if LLM_READY:
    selection, chosen = {}, {}
    for config in CONFIGS:
        runs = [first_choice(config, k) for k in (1, 2, 3)]
        chosen[config] = [[names for names, _ in run] for run in runs]
        acc = [sum(right(r, names) for r, (names, _) in
                   zip(tool_reqs, run)) / len(tool_reqs) for run in runs]
        meters = [m for run in runs for _, m in run]
        selection[config] = {
            "accuracy": acc,
            "tokens_in": sum(m.tokens_in for m in meters) / len(meters),
            "calls": sum(m.calls for m in meters) / len(meters),
            "cost": sum(m.cost() for m in meters),
            "misses": Counter(
                (r["tool"], names[0] if names else "no tool")
                for run in runs for r, (names, _) in zip(tool_reqs, run)
                if not right(r, names))}
    print(f"\n{'configuration':<19}{'accuracy (3 runs)':<21}"
          f"{'calls':>6}{'tokens in':>11}")
    for config, s in selection.items():
        print(f"{config:<19}"
              f"{' '.join(f'{a:.2f}' for a in s['accuracy']):<21}"
              f"{s['calls']:>6.1f}{s['tokens_in']:>11,.0f}")
    core = [t.name for t in ALL_TOOLS]
    print(f"\naccuracy per tool, 3 runs pooled ({3 * 12} requests each):")
    short = ["search", "route", "days", "add", "list", "delete", "remind"]
    print(f"{'':<19}" + "".join(f"{n:>7}" for n in short))
    def per_tool(runs, name):
        hits = [right(r, run[i]) for run in runs
                for i, r in enumerate(tool_reqs) if r["tool"] == name]
        return sum(hits) / len(hits)
    for config, runs in chosen.items():
        print(f"{config:<19}" + "".join(f"{per_tool(runs, n):>7.2f}"
                                        for n in core))
    print("\nmost frequent misses (right tool → chosen), all runs:")
    for config, s in selection.items():
        top = ", ".join(f"{a}→{b} ×{k}"
                        for (a, b), k in s["misses"].most_common(3))
        print(f"  {config:<19}{top or '—'}")
    total = sum(s["cost"] for s in selection.values())
    print(f"\n{3 * len(CONFIGS) * len(tool_reqs)} requests, ${total:.3f}")
    pathlib.Path("selection.json").write_text(json.dumps(
        {c: {k: v for k, v in s.items() if k != "misses"}
         for c, s in selection.items()}), encoding="utf-8")
    pathlib.Path("selection-choices.json").write_text(json.dumps(
        {"ids": [r["id"] for r in tool_reqs], "chosen": chosen}),
        encoding="utf-8")

## Step 3 — Five topologies on one clock

[2. Choose how the work flows and who decides](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/02-choose-how-the-work-flows-and-who-decides/)

In [ ]:
# Topologies on a timeline: the same two specialists in order, in
# parallel, under a supervisor, and in a loop. Latencies are set values
# (a simulation); the graphs and the clock are real.
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send

COST = {"supervisor": 0.25, "router": 0.25, "researcher": 0.6,
        "keeper": 0.4, "merge": 0.3, "writer": 0.4, "critic": 0.2}

class Run(TypedDict, total=False):
    log: Annotated[list, operator.add]     # (agent, start, end)
    rounds: int
    t0: float

def timed(name, update=None):
    """A node that 'thinks' for COST[name] seconds and logs when."""
    def node(s):
        start = time.perf_counter() - s["t0"]
        time.sleep(COST[name])
        end = time.perf_counter() - s["t0"]
        return {"log": [(name, start, end)], **(update(s) if update else {})}
    return node

def order_graph():
    g = StateGraph(Run)
    for n in ("researcher", "keeper", "merge"):
        g.add_node(n, timed(n))
    g.add_edge(START, "researcher"); g.add_edge("researcher", "keeper")
    g.add_edge("keeper", "merge"); g.add_edge("merge", END)
    return g

def parallel_graph():
    g = StateGraph(Run)
    for n in ("router", "researcher", "keeper", "merge"):
        g.add_node(n, timed(n))
    g.add_edge(START, "router")
    g.add_conditional_edges("router", lambda s: [
        Send("researcher", s), Send("keeper", s)], ["researcher", "keeper"])
    g.add_edge(["researcher", "keeper"], "merge")     # waits for both
    g.add_edge("merge", END)
    return g

def supervised_graph(both_at_once):
    plan = [["researcher", "keeper"]] if both_at_once else [
        ["researcher"], ["keeper"]]
    def supervisor(s) -> Command:
        out = timed("supervisor")(s)
        turn = sum(1 for n, *_ in s.get("log", []) if n == "supervisor")
        if turn < len(plan):                         # delegate, then return
            return Command(update=out, goto=[Send(n, s) for n in plan[turn]])
        return Command(update=out, goto=END)         # write the answer
    g = StateGraph(Run)
    g.add_node("supervisor", supervisor)
    for n in ("researcher", "keeper"):
        g.add_node(n, timed(n)); g.add_edge(n, "supervisor")
    g.add_edge(START, "supervisor")
    return g

def loop_graph(max_rounds=3, approved_at=2):
    g = StateGraph(Run)
    g.add_node("writer", timed("writer",
                               lambda s: {"rounds": s.get("rounds", 0) + 1}))
    g.add_node("critic", timed("critic"))
    g.add_edge(START, "writer"); g.add_edge("writer", "critic")
    g.add_conditional_edges("critic", lambda s: END if s["rounds"] in (
        approved_at, max_rounds) else "writer", ["writer", END])
    return g

def bar(start, end, scale=20):
    return " " * round(start * scale) + "█" * max(1, round((end - start)
                                                         * scale))

TOPOLOGIES = {
    "in order": order_graph(),
    "in parallel, merged": parallel_graph(),
    "supervisor, one by one": supervised_graph(False),
    "supervisor, both at once": supervised_graph(True),
    "writer and critic loop": loop_graph(),
}
timing = {}
for name, g in TOPOLOGIES.items():
    t0 = time.perf_counter()
    log = g.compile().invoke({"t0": t0})["log"]
    wall = time.perf_counter() - t0
    timing[name] = {"calls": len(log), "wall": wall,
                    "busy": sum(e - s for _, s, e in log)}
    print(f"{name}: {len(log)} model calls, wall {wall:.2f} s, "
          f"sum of all calls {timing[name]['busy']:.2f} s")
    for who, s, e in sorted(log, key=lambda x: x[1]):
        print(f"  {who:<11}{s:5.2f}–{e:4.2f} {bar(s, e)}")

## Step 4 — A supervisor with specialists as tools

[3. Specialists as tools: the supervisor](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/03-specialists-as-tools-the-supervisor/)

In [ ]:
# A supervisor whose specialists are tools: who calls whom, what each
# one reads, and what comes back.
from langchain.agents import AgentState

TODAY_LINE = f"Today is {TODAY:%A, %Y-%m-%d}."
RESEARCHER = ("You answer questions about the course from the course "
              "notes. Search the notes, then answer in at most three "
              "sentences and cite each claim as (Lecture N, M.K).")
KEEPER = ("You manage the student's to-do list and email reminders. "
          f"{TODAY_LINE} Do only what the request asks. Never invent a "
          "title, a time or a recipient. Say what you did.")
SUPERVISOR = ("You are a study assistant with two specialists. "
              f"{TODAY_LINE} Send questions about the course to "
              "ask_researcher, and to-do, reminder or date requests to "
              "ask_todo_keeper. Give each only its part of the request, "
              "in the student's words. Then answer the student and keep "
              "every citation (Lecture N, M.K) the researcher gave.")

def make_specialists(model_for, keeper_middleware=()):
    researcher = create_agent(model_for("researcher"), NOTES_TOOLS,
                              system_prompt=RESEARCHER, name="researcher",
                              context_schema=Student)
    keeper = create_agent(model_for("keeper"), TODO_TOOLS,
                          system_prompt=KEEPER, name="todo_keeper",
                          middleware=list(keeper_middleware),
                          context_schema=Student)
    return researcher, keeper

def as_tools(researcher, keeper):
    """Each specialist behind a tool: it sees only `request`; the
    supervisor sees only the specialist's last message."""
    @tool
    def ask_researcher(request: str, runtime: ToolRuntime[Student]) -> str:
        """Ask the notes researcher a question about the course. It
        searches the notes and answers with citations (Lecture N, M.K)."""
        out = researcher.invoke({"messages": [HumanMessage(request)]},
                                context=runtime.context)
        return out["messages"][-1].text

    @tool
    def ask_todo_keeper(request: str, runtime: ToolRuntime[Student]) -> str:
        """Ask the to-do keeper to add, list or delete tasks, send an
        email reminder, or count days to a date."""
        out = keeper.invoke({"messages": [HumanMessage(request)]},
                            context=runtime.context)
        return out["messages"][-1].text
    return [ask_researcher, ask_todo_keeper]

def make_supervisor(model, tools, **kw):
    return create_agent(model, tools, system_prompt=SUPERVISOR,
                        context_schema=Student, **kw)

# Scripted replies for one mixed request.
ASK = ("What does few-shot prompting change? Also add 'read Lecture 7' "
       "to my list.")
def cite(messages):              # the researcher's answer from its search
    hit = json.loads(last_tool_text(messages))[0]
    return (f"{hit['text'][:70]}… (Lecture {hit['lecture']}, "
            f"{hit['heading'].split()[0]})")
def combine(messages):           # the supervisor's answer: this turn's
    turn = max(i for i, m in enumerate(messages)        # tool results
               if isinstance(m, HumanMessage))
    return " ".join(str(m.content) for m in messages[turn:]
                    if isinstance(m, ToolMessage))
SCRIPTS = {
    "supervisor": [[("ask_researcher",
                     {"request": "What does few-shot prompting change?"}),
                    ("ask_todo_keeper",
                     {"request": "Add 'read Lecture 7' to my list."})],
                   combine],
    "researcher": [[("search_notes", {"query": "few-shot prompting"})],
                   cite],
    "keeper": [[("add_task", {"title": "read Lecture 7"})],
               lambda m: last_tool_text(m)],
}
models = {role: ScriptedModel(replies=r) for role, r in SCRIPTS.items()}
researcher, keeper = make_specialists(models.get)
supervisor = make_supervisor(models["supervisor"],
                             as_tools(researcher, keeper))
student = Student()
team_out = supervisor.invoke({"messages": [HumanMessage(ASK)]},
                             context=student)

def read_tokens(model, tools):   # messages it read + tool specs per call
    return sum(model.seen) + model.calls * sum(
        approx_tokens(spec(t)) for t in tools)
print("what the supervisor's conversation holds:")
for m in team_out["messages"]:
    what = (", ".join(c["name"] for c in m.tool_calls)
            if getattr(m, "tool_calls", None) else str(m.content)[:60])
    print(f"  {type(m).__name__:<13}{what}")
roles = {"supervisor": as_tools(researcher, keeper),
         "researcher": NOTES_TOOLS, "keeper": TODO_TOOLS}
print(f"\n{'agent':<12}{'model calls':>12}{'≈tokens read':>14}")
for role, tools in roles.items():
    print(f"{role:<12}{models[role].calls:>12}"
          f"{read_tokens(models[role], tools):>14}")
team_calls = sum(m.calls for m in models.values())
team_tokens = sum(read_tokens(models[r], t) for r, t in roles.items())

# The same request handled by the single agent of Lectures 6–9.
single_model = ScriptedModel(replies=[
    [("search_notes", {"query": "few-shot prompting"}),
     ("add_task", {"title": "read Lecture 7"})], combine])
single = create_agent(single_model, ALL_TOOLS, context_schema=Student,
                      system_prompt=RESEARCHER + " " + KEEPER)
single.invoke({"messages": [HumanMessage(ASK)]}, context=Student())
single_tokens = read_tokens(single_model, ALL_TOOLS)
print(f"{'one agent':<12}{single_model.calls:>12}{single_tokens:>14}")
print(f"the team: {team_calls / single_model.calls:.0f}× the model calls, "
      f"{team_tokens / single_tokens:.1f}× the tokens on this request")
print("tasks after the run:", list(student.tasks.values()))

# What a specialist receives: only the request, or the whole conversation.
history = [HumanMessage(q["question"]) for q in val[:6]] + [
    AIMessage(" ".join(n["text"].split()[:120]))
    for n in retrieve("few-shot prompting", 6)]
request = "What does few-shot prompting change?"
isolated = approx_tokens(request)
forked = sum(approx_tokens(str(m.content)) for m in history) + isolated
print(f"\nthe researcher's input after 6 earlier turns: the request alone "
      f"≈{isolated} tokens; with the conversation ≈{forked} tokens")

# A specialist can also return data: a Command updates the supervisor's
# state (here, a list of citations) next to the ToolMessage.
import operator
from typing import Annotated
class TeamState(AgentState):
    sources: Annotated[list, operator.add]     # parallel calls append

@tool
def ask_researcher_data(request: str, runtime: ToolRuntime[Student]):
    """Ask the notes researcher; its citations are kept as data."""
    out = researcher.invoke({"messages": [HumanMessage(request)]},
                            context=runtime.context)
    answer = out["messages"][-1].text
    found = re.findall(r"\(Lecture (\d), (\d+\.\d+)\)", answer)
    return Command(update={
        "sources": [f"L{a} {b}" for a, b in found],
        "messages": [ToolMessage(answer, tool_call_id=runtime.tool_call_id)]})

models = {role: ScriptedModel(replies=r) for role, r in SCRIPTS.items()}
models["supervisor"] = ScriptedModel(replies=[
    [("ask_researcher_data", {"request": request})], combine])
researcher, keeper = make_specialists(models.get)
sup_data = create_agent(models["supervisor"], [ask_researcher_data],
                        state_schema=TeamState, context_schema=Student)
out = sup_data.invoke({"messages": [HumanMessage(request)]},
                      context=Student())
print("supervisor state keys:", sorted(out), "| sources:", out["sources"])

## Step 5 — Many specialists: one dispatch tool and a registry

[3. Specialists as tools: the supervisor](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/03-specialists-as-tools-the-supervisor/)

In [ ]:
# Many specialists: one tool each, one dispatch tool with a list, or a
# dispatch tool plus a search over a registry.
REGISTRY = {   # name: (description, the agent; None = not built here)
    "researcher": ("Answers questions about the course from the notes, "
                   "with citations.", researcher),
    "todo_keeper": ("Adds, lists and deletes tasks; sends email "
                    "reminders; counts days to a date.", keeper),
    "quiz_maker": ("Writes practice questions from the course question "
                   "set of Lecture 3.", None),
    "flashcard_coach": ("Turns a section into revision flashcards and "
                        "schedules reviews.", None),
    "glossary_explainer": ("Explains a course term with its glossary "
                           "entry and an example.", None),
    "code_tutor": ("Explains and debugs the student's notebook code.",
                   None),
    "calendar_planner": ("Plans study sessions in the student's "
                         "calendar.", None),
    "forum_summarizer": ("Summarises the course forum threads on a "
                         "topic.", None),
    "grade_reporter": ("Reports the student's quiz and homework "
                       "results.", None),
    "paper_reader": ("Reads a research paper and explains its method.",
                     None),
    "translator_uk": ("Translates course text into Ukrainian with the "
                      "course glossary.", None),
    "progress_coach": ("Reviews what the student has read and suggests "
                       "what to study next.", None),
}

def ask_tool(name, description):          # (a) one tool per specialist
    return stub(f"ask_{name}", description, create_model(
        name, request=(str, ...)))
one_each = [ask_tool(n, d) for n, (d, _) in REGISTRY.items()]

@tool
def task(agent: str, request: str, runtime: ToolRuntime[Student]) -> str:
    """Give a request to one specialist and return its answer.

    Specialists:
    {agents}
    """
    if agent not in REGISTRY:
        return (f"Error: no specialist named {agent!r}. Use one of: "
                + ", ".join(REGISTRY))
    _, specialist = REGISTRY[agent]
    if specialist is None:
        return f"({agent} is not built in this lecture.)"
    out = specialist.invoke({"messages": [HumanMessage(request)]},
                            context=runtime.context)
    return out["messages"][-1].text
listing = "\n    ".join(f"{n}: {d}" for n, (d, _) in REGISTRY.items())
task_with_list = task.model_copy(update={
    "description": task.description.format(agents=listing)})
task_alone = task.model_copy(update={
    "description": task.description.split("\n")[0] + " Find the "
    "specialist's name with search_agents first."})

@tool
def search_agents(query: str) -> list[str]:
    """Find specialists for a job: returns the 3 best names with what
    each does."""
    names = list(REGISTRY)
    docs = [n.replace("_", " ") + " " + REGISTRY[n][0] for n in names]
    return [f"{names[i]}: {REGISTRY[names[i]][0]}"
            for i in word_search(docs, query, 3)]

designs = {"one tool per specialist": one_each,
           "task + list in its description": [task_with_list],
           "task + search_agents": [task_alone, search_agents]}
print(f"{'12 specialists':<32}{'≈tokens per supervisor call':>28}")
for name, tools in designs.items():
    print(f"{name:<32}{sum(approx_tokens(spec(t)) for t in tools):>28}")
found = search_agents.invoke({"query": "remind me about a deadline"})
print("search_agents('remind me about a deadline') →")
for line in found:
    print("  ", line)
print(f"  ≈{approx_tokens(found)} tokens, paid only by the calls that "
      "search")

# The dispatch loop with a scripted supervisor: search, a wrong name,
# the error, the corrected call.
models = {role: ScriptedModel(replies=r) for role, r in SCRIPTS.items()}
researcher, keeper = make_specialists(models.get)
REGISTRY["researcher"] = (REGISTRY["researcher"][0], researcher)
REGISTRY["todo_keeper"] = (REGISTRY["todo_keeper"][0], keeper)
dispatcher_model = ScriptedModel(replies=[
    [("search_agents", {"query": "add a task to my list"})],
    [("task", {"agent": "todo-keeper",
               "request": "Add 'read Lecture 7' to my list."})],
    [("task", {"agent": "todo_keeper",
               "request": "Add 'read Lecture 7' to my list."})],
    lambda m: last_tool_text(m)])
dispatcher = create_agent(dispatcher_model, [task_alone, search_agents],
                          context_schema=Student)
out = dispatcher.invoke({"messages": [HumanMessage(
    "Add 'read Lecture 7' to my list.")]}, context=Student())
for m in out["messages"][1:]:
    if getattr(m, "tool_calls", None):
        c = m.tool_calls[0]
        print(f"  call  {c['name']}({json.dumps(c['args'])[:58]})")
    else:
        print(f"  {'tool' if isinstance(m, ToolMessage) else 'reply':<6}"
              f"{str(m.content)[:66]}")

## Step 6 — Handoffs, and the message pair that must survive

[4. Hand the conversation over, or route it](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/04-hand-the-conversation-over-or-route-it/)

In [ ]:
# Handoffs: the specialist that was asked takes over the conversation.
# Two implementations, the message pair that must survive, and the cost
# per turn compared with a supervisor.
from typing import NotRequired
from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState

class HandoffState(AgentState):
    active_agent: NotRequired[str]

def transfer(to):
    @tool(f"transfer_to_{to}", description=f"Hand the conversation to the "
          f"{to.replace('_', ' ')} when the student's request is its job.")
    def handoff(runtime: ToolRuntime[Student]) -> Command:
        return Command(update={"active_agent": to, "messages": [
            ToolMessage(f"Transferred to {to}.",
                        tool_call_id=runtime.tool_call_id)]})
    return handoff

ROLES = {"researcher": (RESEARCHER, NOTES_TOOLS + [transfer("todo_keeper")]),
         "todo_keeper": (KEEPER, TODO_TOOLS + [transfer("researcher")])}
acted = []                          # which role made each model call

@wrap_model_call
def act_as(request, handler):
    """One agent, two roles: the prompt and tools follow active_agent."""
    role = request.state.get("active_agent", "researcher")
    acted.append(role)
    prompt, tools = ROLES[role]
    return handler(request.override(system_message=SystemMessage(prompt),
                                    tools=tools))

TURNS = ["What does few-shot prompting change?",
         "Add that to my list as 'revise few-shot prompting'.",
         "And email me a reminder tomorrow at 9:00."]
REMIND = {"text": "revise few-shot prompting", "when": "2026-11-06 09:00"}
one_model = ScriptedModel(replies=[
    [("search_notes", {"query": "few-shot prompting"})], cite,
    [("transfer_to_todo_keeper", {})],
    [("add_task", {"title": "revise few-shot prompting"})], last_tool_text,
    [("send_reminder", REMIND)], last_tool_text])
all_tools = [t for _, ts in ROLES.values() for t in ts]
switching = create_agent(one_model, all_tools, middleware=[act_as],
                         state_schema=HandoffState, context_schema=Student,
                         checkpointer=InMemorySaver())

def per_turn(app, models, cfg):
    """Model calls made in each turn of the same conversation."""
    student, counts = Student(), []
    for text in TURNS:
        before = sum(m.calls for m in models)
        out = app.invoke({"messages": [HumanMessage(text)]}, cfg,
                         context=student)
        counts.append(sum(m.calls for m in models) - before)
    return counts, out, student

handoff_calls, out, student = per_turn(
    switching, [one_model], {"configurable": {"thread_id": "handoff"}})
print("handoffs, one agent that switches roles:")
print("  roles per model call:", " ".join(acted))
print("  active after the conversation:", out["active_agent"],
      "| outbox:", student.outbox)

# The same conversation through the supervisor of Step 4.
sup_models = {
    "supervisor": ScriptedModel(replies=[
        [("ask_researcher", {"request": TURNS[0]})], combine,
        [("ask_todo_keeper",
          {"request": "Add 'revise few-shot prompting' to my list."})],
        combine,
        [("ask_todo_keeper", {"request": "Email me a reminder tomorrow "
                              "at 9:00 to revise few-shot prompting."})],
        combine]),
    "researcher": ScriptedModel(replies=[
        [("search_notes", {"query": "few-shot prompting"})], cite]),
    "keeper": ScriptedModel(replies=[
        [("add_task", {"title": "revise few-shot prompting"})],
        last_tool_text, [("send_reminder", REMIND)], last_tool_text])}
researcher, keeper = make_specialists(sup_models.get)
sup = make_supervisor(sup_models["supervisor"], as_tools(researcher, keeper),
                      checkpointer=InMemorySaver())
sup_calls, _, _ = per_turn(sup, list(sup_models.values()),
                           {"configurable": {"thread_id": "supervisor"}})
print(f"\n{'turn':<56}{'handoffs':>9}{'supervisor':>11}")
for text, h, s in zip(TURNS, handoff_calls, sup_calls):
    print(f"{text:<56}{h:>9}{s:>11}")
print(f"{'model calls in total':<56}{sum(handoff_calls):>9}"
      f"{sum(sup_calls):>11}")

# The second implementation: each agent is a node of a parent graph, and
# a handoff tool jumps to the other node with Command.PARENT. The parent
# receives only what the update carries, so it must carry the AIMessage
# with the call AND the ToolMessage that answers it.
class Desk(MessagesState):
    active_agent: str

def node_transfer(to, with_answer=True):
    @tool(f"transfer_to_{to}", description=f"Hand the conversation to the "
          f"{to.replace('_', ' ')}.")
    def handoff(runtime: ToolRuntime[Student]) -> Command:
        call = next(m for m in reversed(runtime.state["messages"])
                    if isinstance(m, AIMessage))
        answer = ToolMessage(f"Transferred to {to}.",
                             tool_call_id=runtime.tool_call_id)
        return Command(goto=to, graph=Command.PARENT, update={
            "active_agent": to,
            "messages": [call, answer] if with_answer else [call]})
    return handoff

def build_desk(with_answer):
    models = {"researcher": ScriptedModel(replies=[
                  [("transfer_to_todo_keeper", {})]]),
              "keeper": ScriptedModel(replies=[
                  [("add_task", {"title": "revise few-shot prompting"})],
                  last_tool_text])}
    researcher_node = create_agent(
        models["researcher"],
        NOTES_TOOLS + [node_transfer("todo_keeper", with_answer)],
        system_prompt=RESEARCHER, context_schema=Student)
    keeper_node = create_agent(models["keeper"], TODO_TOOLS,
                               system_prompt=KEEPER, context_schema=Student)
    g = StateGraph(Desk, context_schema=Student)
    g.add_node("researcher", researcher_node)
    g.add_node("todo_keeper", keeper_node)
    g.add_conditional_edges(START, lambda s: s.get("active_agent",
                                                   "researcher"),
                            ["researcher", "todo_keeper"])
    g.add_edge("researcher", END); g.add_edge("todo_keeper", END)
    return g.compile()

def unanswered(messages):
    """Tool calls in a history that no ToolMessage answers."""
    answered = {m.tool_call_id for m in messages
                if isinstance(m, ToolMessage)}
    return [c["id"] for m in messages if isinstance(m, AIMessage)
            for c in m.tool_calls if c["id"] not in answered]

for with_answer in (True, False):
    out = build_desk(with_answer).invoke(
        {"messages": [HumanMessage(TURNS[1])]}, context=Student())
    label = "with the ToolMessage" if with_answer else "call only"
    print(f"\nnodes, handoff {label}: active {out['active_agent']}, "
          f"unanswered calls {unanswered(out['messages'])}")
    for m in out["messages"]:
        what = (", ".join(c["name"] for c in m.tool_calls)
                if getattr(m, "tool_calls", None) else str(m.content)[:50])
        print(f"  {type(m).__name__:<13}{what}")
broken = out["messages"]

# 💳 A provider checks the pair: the broken history, sent as it is.
if LLM_READY:
    try:
        hosted.invoke(broken)
        print("\nthe hosted model accepted the broken history")
    except Exception as e:
        print(f"\nthe hosted model refused it: {type(e).__name__}: "
              f"{str(e)[:160]}")

# What the next agent reads: the whole history, or a short handover.
long_history = [HumanMessage(q["question"]) for q in val[:8]] + [
    AIMessage(" ".join(n["text"].split()[:120]))
    for n in retrieve("few-shot prompting", 8)]
handover = [HumanMessage(TURNS[1]),
            AIMessage("Context: the student asked what few-shot "
                      "prompting changes; the notes answer in Lecture 4, "
                      "2.1.")]
for name, msgs in (("whole history", long_history + handover),
                   ("short handover", handover)):
    print(f"{name:<16}{len(msgs):>3} messages, "
          f"≈{sum(approx_tokens(str(m.content)) for m in msgs):>5} tokens")

## Step 7 — A router that sends parts to several specialists

[4. Hand the conversation over, or route it](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/04-hand-the-conversation-over-or-route-it/)

In [ ]:
# A router: one classification, then the specialists it names run in
# parallel (Send), each on its own part of the message; code merges.
from langgraph.runtime import Runtime

class Part(BaseModel):
    specialist: Literal["researcher", "todo_keeper"]
    request: str = Field(description="the part of the student's message "
                         "for this specialist, in the student's words")
class Routes(BaseModel):
    """Which specialists the message needs: none, one or several."""
    parts: list[Part]

ROUTER = ("Split the student's message into parts for the specialists. "
          "researcher: questions about the course. todo_keeper: tasks, "
          "reminders, dates. One part per specialist needed; none if "
          "neither fits. " + TODAY_LINE)

class Desk2(TypedDict, total=False):
    message: str
    parts: list
    answers: Annotated[list, operator.add]      # parallel writes append
    answer: str

def make_router_team(classify, researcher, keeper):
    agents = {"researcher": researcher, "todo_keeper": keeper}
    def route(s):
        return {"parts": classify(s["message"]).parts}
    def fan_out(s):
        return [Send("specialist", {"parts": [p]}) for p in s["parts"]] \
            or [Send("merge", s)]
    def specialist(s, runtime: Runtime[Student]):
        p = s["parts"][0]
        out = agents[p.specialist].invoke(
            {"messages": [HumanMessage(p.request)]}, context=runtime.context)
        return {"answers": [out["messages"][-1].text]}
    def merge(s):
        return {"answer": "\n\n".join(s.get("answers", []))
                or "I can help with the course notes and your to-do list."}
    g = StateGraph(Desk2, context_schema=Student)
    g.add_node(route); g.add_node(specialist); g.add_node(merge)
    g.add_edge(START, "route")
    g.add_conditional_edges("route", fan_out, ["specialist", "merge"])
    g.add_edge("specialist", "merge"); g.add_edge("merge", END)
    return g.compile()

TODO_WORDS = re.compile(r"\b(add|put|delete|remove|remind|reminder|task|"
                        r"tasks|list|days|email me)\b", re.I)
def classify_offline(message):
    """Stand-in classifier: each sentence goes to the to-do keeper if it
    has a to-do word, otherwise to the researcher."""
    parts = defaultdict(list)
    for sentence in re.split(r"(?<=[.?!])\s+", message.strip()):
        who = "todo_keeper" if TODO_WORDS.search(sentence) else "researcher"
        parts[who].append(sentence)
    return Routes(parts=[Part(specialist=k, request=" ".join(v))
                         for k, v in parts.items()])

NEEDS = {"notes": {"researcher"}, "todo": {"todo_keeper"},
         "mixed": {"researcher", "todo_keeper"}}
team_reqs = [r for r in requests if r["set"] == "team"]
print(f"{'kind':<7}{'requests':>9}{'one route':>11}{'every route':>13}")
for kind, needed in NEEDS.items():
    rs = [r for r in team_reqs if r["kind"] == kind]
    got = [{p.specialist for p in classify_offline(r["request"]).parts}
           for r in rs]
    first = [{classify_offline(r["request"]).parts[0].specialist}
             for r in rs]
    print(f"{kind:<7}{len(rs):>9}{sum(f >= needed for f in first):>11}"
          f"{sum(g >= needed for g in got):>13}")

# One mixed request through the graph, with scripted specialists.
models = {role: ScriptedModel(replies=r) for role, r in SCRIPTS.items()}
researcher, keeper = make_specialists(models.get)
desk = make_router_team(classify_offline, researcher, keeper)
for mode, event in desk.stream({"message": ASK}, context=Student(),
                               stream_mode=["debug", "values"]):
    if mode == "values":
        out = event                                  # the latest state
    elif event["type"] == "task":
        print(f"  super-step {event['step']}: {event['payload']['name']}")
print("merged answer:", out["answer"].replace("\n\n", " | ")[:110])
print("model calls:", sum(m.calls for m in models.values()),
      "(no model writes the merged answer)")

## Step 8 — Run a specialist in the background

[5. Run specialists safely](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/05-run-specialists-safely/)

In [ ]:
# A specialist in the background: start returns a job id at once; the
# supervisor keeps talking; the student can check or cancel the job.
JOBS = {}
def start(agent, request, context):
    """Run an agent in a thread; return a job id at once. The job checks
    for a cancel between two steps of the agent's loop."""
    job_id = f"job-{len(JOBS) + 1}"
    job = JOBS[job_id] = {"status": "pending", "done": 0, "result": None,
                          "cancel": threading.Event()}
    def run():
        job["status"] = "running"
        for step in agent.stream({"messages": [HumanMessage(request)]},
                                 context=context, stream_mode="updates"):
            if "tools" in step:
                job["done"] += len(step["tools"]["messages"])
            if job["cancel"].is_set():
                job["status"] = "cancelled"
                return
            if "model" in step:
                job["result"] = step["model"]["messages"][-1].text
        job["status"] = "completed"
    threading.Thread(target=run, daemon=True).start()
    return job_id

@tool
def read_section(heading: str) -> str:
    """Read one whole section of the notes (slow)."""
    time.sleep(0.3)
    return f"read {heading}"

TOPIC = [n["heading"] for n in retrieve("attention", 10)
         if NUMBERED.match(n["heading"])][:4]
def slow_researcher():
    return create_agent(ScriptedModel(replies=[
        [("read_section", {"heading": h})] for h in TOPIC] + [
        f"Attention is covered in {len(TOPIC)} sections: "
        + "; ".join(TOPIC) + "."]), [read_section])

@tool
def start_research(request: str, runtime: ToolRuntime[Student]) -> str:
    """Start a long research job in the background; returns its id."""
    return f"Started {start(slow_researcher(), request, runtime.context)}."

@tool
def check_job(job_id: str) -> str:
    """Status of a background job, and its result once completed."""
    job = JOBS[job_id]
    return (f"{job_id}: {job['status']}, {job['done']} of {len(TOPIC)} "
            f"sections read" + (f". Result: {job['result']}"
                                if job["result"] else ""))

@tool
def cancel_job(job_id: str) -> str:
    """Cancel a background job; it stops before its next step."""
    JOBS[job_id]["cancel"].set()
    return f"{job_id}: cancel requested"

models = {role: ScriptedModel(replies=r) for role, r in SCRIPTS.items()}
models["keeper"] = ScriptedModel(replies=[
    [("add_task", {"title": "revise attention"})], last_tool_text])
_, keeper = make_specialists(models.get)
ask_keeper = as_tools(researcher, keeper)[1]
supervisor_model = ScriptedModel(replies=[
    [("start_research", {"request": "Summarise what the notes say about "
                         "attention."}),
     ("ask_todo_keeper", {"request": "Add 'revise attention' to my list."})],
    combine,
    [("check_job", {"job_id": "job-1"})], last_tool_text,
    [("check_job", {"job_id": "job-1"})], last_tool_text,
    [("start_research", {"request": "Summarise what the notes say about "
                         "LoRA."})], combine,
    [("cancel_job", {"job_id": "job-2"})], last_tool_text,
    [("check_job", {"job_id": "job-2"})], last_tool_text])
front = create_agent(supervisor_model,
                     [start_research, check_job, cancel_job, ask_keeper],
                     context_schema=Student, checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "background"}}
student, t0 = Student(), time.perf_counter()
def say(text, wait_until=None):
    if wait_until:
        time.sleep(max(0, wait_until - (time.perf_counter() - t0)))
    out = front.invoke({"messages": [HumanMessage(text)]}, cfg,
                       context=student)
    print(f"{time.perf_counter() - t0:5.2f} s  student: {text}")
    print(f"{'':9}assistant: {out['messages'][-1].text[:100]}")
say("Summarise what the notes say about attention, and add 'revise "
    "attention' to my list.")
say("Is the summary ready?", wait_until=0.5)
say("And now?", wait_until=1.6)
say("Summarise what the notes say about LoRA.", wait_until=1.7)
say("Never mind, cancel it.", wait_until=1.8)
say("Did it stop?", wait_until=2.4)
print("tasks:", list(student.tasks.values())[-1:], "| jobs:",
      {k: v["status"] for k, v in JOBS.items()})

## Step 9 — Pause a specialist for approval, resume through the supervisor

[5. Run specialists safely](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/05-run-specialists-safely/)

In [ ]:
# Approval inside a specialist: the to-do keeper pauses before an
# irreversible action; the pause surfaces through the supervisor and is
# answered on the supervisor's thread.
from langchain.agents.middleware import HumanInTheLoopMiddleware

gate = HumanInTheLoopMiddleware(        # Lecture 9's policy: these two
    interrupt_on={"delete_task": True, "send_reminder": True},  # need a yes
    interrupt_mode="per_call")          # one pause per call, keyed by id
FLOUR = "Delete the bread task and email me a reminder tomorrow at 9:00 " \
        "to buy flour."
models = {
    "supervisor": ScriptedModel(replies=[
        [("ask_todo_keeper", {"request": FLOUR})], combine]),
    "keeper": ScriptedModel(replies=[
        [("list_tasks", {})],
        [("delete_task", {"task_id": "t3"}),
         ("send_reminder", {"text": "buy flour",
                            "when": "2026-11-06 09:00"})],
        lambda m: "Done: " + " ".join(str(x.content) for x in m[-2:])]),
    "researcher": ScriptedModel(replies=["(not used)"])}
researcher, keeper = make_specialists(models.get, keeper_middleware=[gate])
boss = make_supervisor(models["supervisor"], as_tools(researcher, keeper),
                       checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "student-42/flour"}}
student = Student()
out = boss.invoke({"messages": [HumanMessage(FLOUR)]}, cfg,
                  context=student, version="v2")
print(f"paused: {len(out.interrupts)} questions for the student")
for it in out.interrupts:
    v = it.value
    print(f"  interrupt {it.id[:8]}…: {v['type']}, {v['name']}({v['args']})")
print("tasks now:", list(student.tasks), "| outbox now:", student.outbox)
print("keeper model calls so far:", models["keeper"].calls)

# The student approves the deletion and moves the reminder to 8:00.
answers = {}
for it in out.interrupts:
    if it.value["name"] == "delete_task":
        answers[it.id] = {"type": "approve"}
    else:
        answers[it.id] = {"type": "edit", "edited_action": {
            "name": "send_reminder",
            "args": {**it.value["args"], "when": "2026-11-06 08:00"}}}
out = boss.invoke(Command(resume=answers), cfg, context=student,
                  version="v2")
print("resumed; pauses left:", len(out.interrupts))
print("tasks now:", list(student.tasks), "| outbox now:", student.outbox)
print("supervisor's answer:", out.value["messages"][-1].text[:110])
print("model calls: supervisor", models["supervisor"].calls,
      "| keeper", models["keeper"].calls)

## Step 10 — Compare one agent, a supervisor and a router 💳 API

[6. Measure whether the team beats one agent](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/06-measure-whether-the-team-beats-one-agent/)

In [ ]:
# 💳 One agent, a supervisor and a router on the same 60 requests, three
# runs each: task success, model calls, tokens, cost, latency, and why
# each failure happened.
import ast
SINGLE = (f"You are a study assistant. {TODAY_LINE} For questions about "
          "the course, search the notes, answer in at most three "
          "sentences and cite each claim as (Lecture N, M.K). For the "
          "to-do list and email reminders, do only what the request "
          "asks; never invent a title, a time or a recipient.")

class Trace(Meter):
    """A Meter that also keeps every tool call and tool output."""
    def __init__(self):
        super().__init__()
        self.tools, self.outputs = [], []
    def on_tool_start(self, serialized, input_str, **kw):
        with self.lock:
            self.tools.append((serialized or {}).get("name") or kw.get("name"))
    def on_tool_end(self, output, **kw):
        with self.lock:
            self.outputs.append(str(getattr(output, "content", output)))

def check(r, answer, student):
    """Deterministic checks of one request: the problems found."""
    problems, e = [], r["expect"]
    if r["kind"] in ("notes", "mixed"):
        number = r["heading"].split()[0]
        if not (f"Lecture {r['lecture']}" in answer and number in answer):
            problems.append("citation")
    first = dict(FIRST_TASKS)
    added = [v.lower() for k, v in student.tasks.items() if k not in first]
    deleted = sorted(k for k in first if k not in student.tasks)
    wanted = e.get("add", [])
    if deleted != sorted(e.get("delete", [])) or len(added) != len(
            wanted) or not all(any(all(w in a for w in words)
                                   for a in added) for words in wanted):
        problems.append("tasks")
    if sorted(x["when"] for x in student.outbox) != sorted(
            e.get("remind", [])):
        problems.append("reminder")
    if any(w.lower() not in answer.lower() for w in e.get("say", [])):
        problems.append("answer")
    return problems

def hits(output):
    """The sections in one search_notes output (JSON or Python text)."""
    for parse in (json.loads, ast.literal_eval):
        try:
            value = parse(output)
        except (ValueError, SyntaxError):
            continue
        return [h for h in value if isinstance(h, dict)] \
            if isinstance(value, list) else []
    return []

def found(r, outputs):
    """Did any search during the run return the gold section?"""
    return any(h.get("lecture") == r["lecture"]
               and h.get("heading") == r["heading"]
               for o in outputs for h in hits(o))

NOTES_NAMES = {"search_notes", "route_question"}
TODO_NAMES = {t.name for t in TODO_TOOLS}
def why(r, problems, trace, answer):
    """The first matching failure class."""
    used = set(trace.tools)
    if r["kind"] in ("notes", "mixed") and not used & NOTES_NAMES or \
            r["kind"] in ("todo", "mixed") and not used & TODO_NAMES:
        return "part not handled"
    if "citation" in problems:
        if found(r, trace.outputs):
            return "found, not cited"
        if f"Lecture {r['lecture']}" in answer:
            return "same lecture, other section"
        return "not found"
    if "reminder" in problems or "tasks" in problems:
        return "wrong action"
    return "wrong answer"

def build(design):
    """A fresh team for a design: (invoke(request, student, cfg) → answer)."""
    researcher, keeper = make_specialists(lambda role: hosted)
    if design == "one agent":
        app = create_agent(hosted, ALL_TOOLS, system_prompt=SINGLE,
                           context_schema=Student)
        return lambda text, student, cfg: app.invoke(
            {"messages": [HumanMessage(text)]}, cfg,
            context=student)["messages"][-1].text
    if design == "supervisor":
        app = make_supervisor(hosted, as_tools(researcher, keeper))
        return lambda text, student, cfg: app.invoke(
            {"messages": [HumanMessage(text)]}, cfg,
            context=student)["messages"][-1].text
    splitter = hosted.with_structured_output(Routes)
    classify = lambda text: splitter.invoke([("system", ROUTER),
                                             ("human", text)])
    app = make_router_team(classify, researcher, keeper)
    return lambda text, student, cfg: app.invoke(
        {"message": text}, cfg, context=student)["answer"]

DESIGNS = ["one agent", "supervisor", "router"]
def run_design(design, run):
    ask = build(design)
    def one(r):
        trace, student, t0 = Trace(), Student(), time.perf_counter()
        try:
            answer = ask(r["request"], student, {"callbacks": [trace]})
        except Exception as e:
            answer = f"error: {type(e).__name__}"
        problems = check(r, answer, student)
        return {"id": r["id"], "kind": r["kind"], "ok": not problems,
                "why": why(r, problems, trace, answer) if problems
                else None,
                "calls": trace.calls, "tokens_in": trace.tokens_in,
                "tokens_out": trace.tokens_out, "cost": trace.cost(),
                "seconds": time.perf_counter() - t0, "answer": answer,
                "tools": trace.tools,
                "gold_found": r["kind"] != "todo" and found(r, trace.outputs)}
    return in_parallel(f"{design}, run {run}", one, team_reqs)

def pct(values, q):
    v = sorted(values)
    return v[min(len(v) - 1, int(q * len(v)))]

if LLM_READY:
    results = {d: [run_design(d, k) for k in (1, 2, 3)] for d in DESIGNS}
    print(f"\n{'design':<12}{'success (3 runs)':<18}{'notes':>6}"
          f"{'to-do':>6}{'mixed':>6}{'calls':>7}{'tokens':>8}"
          f"{'$/1k ok':>9}{'p50 s':>7}{'p95 s':>7}")
    summary = {}
    for d, runs in results.items():
        rows_ = [x for run in runs for x in run]
        ok = [sum(x["ok"] for x in run) / len(run) for run in runs]
        by_kind = {k: sum(x["ok"] for x in rows_ if x["kind"] == k)
                   / sum(x["kind"] == k for x in rows_)
                   for k in ("notes", "todo", "mixed")}
        cost = sum(x["cost"] for x in rows_)
        summary[d] = {
            "success": ok, **by_kind,
            "calls": sum(x["calls"] for x in rows_) / len(rows_),
            "tokens": sum(x["tokens_in"] + x["tokens_out"]
                          for x in rows_) / len(rows_),
            "per_success": cost / max(1, sum(x["ok"] for x in rows_)),
            "p50": pct([x["seconds"] for x in rows_], 0.5),
            "p95": pct([x["seconds"] for x in rows_], 0.95),
            "cost": cost,
            "why": Counter(x["why"] for x in rows_ if x["why"])}
        s = summary[d]
        print(f"{d:<12}{' '.join(f'{a:.2f}' for a in ok):<18}"
              f"{s['notes']:>6.2f}{s['todo']:>6.2f}{s['mixed']:>6.2f}"
              f"{s['calls']:>7.1f}{s['tokens']:>8,.0f}"
              f"{1000 * s['per_success']:>9.2f}{s['p50']:>7.1f}"
              f"{s['p95']:>7.1f}")
    print("\nwhy requests failed (all three runs):")
    for d, s in summary.items():
        print(f"  {d:<12}" + (", ".join(f"{k} {v}" for k, v in
                                        s["why"].most_common()) or "—"))
    flips = {d: sum(len({run[i]["ok"] for run in results[d]}) > 1
                    for i in range(len(team_reqs))) for d in DESIGNS}
    print("requests whose outcome changed between runs:", flips)
    asked = 3 * sum(r["kind"] != "todo" for r in team_reqs)
    print("a search returned the gold section (notes and mixed runs):", {
        d: f"{sum(x['gold_found'] for run in results[d] for x in run)}"
           f"/{asked}" for d in DESIGNS})
    total = sum(s["cost"] for s in summary.values())
    print(f"{3 * len(DESIGNS) * len(team_reqs)} runs, ${total:.3f}")
    pathlib.Path("team.json").write_text(json.dumps(
        {d: {k: v for k, v in s.items() if k != "why"} | {
            "why": dict(s["why"])} for d, s in summary.items()}),
        encoding="utf-8")
    pathlib.Path("team-answers.json").write_text(json.dumps(
        results, ensure_ascii=False, indent=1), encoding="utf-8")

## Step 11 — The lecture in one table

[6. Measure whether the team beats one agent](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-10-fable/06-measure-whether-the-team-beats-one-agent/)

In [ ]:
# The lecture in one table: what each design bought and what it cost,
# from the recorded runs (Steps 2 and 10) and the offline steps.
sel = pathlib.Path("selection.json")
team = pathlib.Path("team.json")
if sel.exists():
    s = json.loads(sel.read_text(encoding="utf-8"))
    print("tool selection (Step 2), mean accuracy of 3 runs:")
    for config, x in s.items():
        acc = sum(x["accuracy"]) / 3
        print(f"  {config:<19}{acc:.2f}  "
              f"(runs {min(x['accuracy']):.2f}–{max(x['accuracy']):.2f}), "
              f"{x['tokens_in']:,.0f} tokens in per request")
else:
    print("tool selection: needs Step 2 (💳)")

print(f"\n{'design':<12}{'success':>8}{'calls':>7}{'$/1k ok':>9}"
      f"{'p95 s':>7}  main failure")
if team.exists():
    t = json.loads(team.read_text(encoding="utf-8"))
    for d, x in t.items():
        main = max(x["why"], key=x["why"].get) if x["why"] else "—"
        print(f"{d:<12}{sum(x['success']) / 3:>8.2f}{x['calls']:>7.1f}"
              f"{1000 * x['per_success']:>9.2f}{x['p95']:>7.1f}  {main}")
else:
    print("needs Step 10 (💳)")

print("\noffline, on one request or conversation:")
print(f"  supervisor vs one agent (Step 4): {team_calls} vs "
      f"{single_model.calls} model calls")
print(f"  handoffs vs supervisor over 3 turns (Step 6): "
      f"{sum(handoff_calls)} vs {sum(sup_calls)} model calls")
print(f"  parallel vs in order (Step 3): "
      f"{timing['in parallel, merged']['wall']:.2f} s vs "
      f"{timing['in order']['wall']:.2f} s wall time")